# Projetoooo

Author: João Victor Colombari Carlet (5274502)

In [1]:
import sys
print(sys.executable)

!{sys.executable} -m pip -q install numpy bleak nest_asyncio asyncio ipympl pyqtgraph PyQt6 PyOpenGL

/Users/joaovitor/Documents/PhD/aulas_USP/VR-AR/yaw-walk-with-me/venv/bin/python

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import nest_asyncio
nest_asyncio.apply()

In [7]:
import asyncio
from bleak import BleakScanner

async def scan():
    devices = await BleakScanner.discover(
        return_adv=True,
        timeout=10
    )

    for addr, (device, adv) in devices.items():
        print("\n================")
        print("ADDRESS:", device.address)
        print("NAME:", device.name)
        print("RSSI:", adv.rssi)
        print("LOCAL NAME:", adv.local_name)
        print("UUIDS:", adv.service_uuids)
        print("MFG DATA:", adv.manufacturer_data)

await scan()


ADDRESS: C04C97BA-ED53-E435-DDE4-7055A36EF461
NAME: Tiresias_DK
RSSI: -47
LOCAL NAME: Tiresias_DK
UUIDS: ['00001523-1212-efde-1523-785feabcd123']
MFG DATA: {}

ADDRESS: 5FAAE2EC-74EB-08C3-6EF0-DA068E957342
NAME: iPhone de Joao victor (3)
RSSI: -64
LOCAL NAME: None
UUIDS: []
MFG DATA: {76: b'\x10\x06;\x1e\xc5m9T'}

ADDRESS: E82D2EA8-0F8E-2AA2-F1B7-A9BF6CD8A5DE
NAME: None
RSSI: -58
LOCAL NAME: None
UUIDS: []
MFG DATA: {76: b'\x12\x02\x00\x03'}

ADDRESS: 83838F79-0A8B-0FE9-A8E5-5D1ADABAD781
NAME: None
RSSI: -58
LOCAL NAME: None
UUIDS: []
MFG DATA: {76: b'\x16\x08\x00j\xf8\xbbO\xb3\x1d\xa2'}


In [ ]:
import asyncio
import struct
import math
import nest_asyncio
import numpy as np

from bleak import BleakScanner, BleakClient

import pyqtgraph as pg
import pyqtgraph.opengl as gl

from PyQt6.QtWidgets import QApplication
from PyQt6.QtCore import QTimer

nest_asyncio.apply()

# ============================================================
# CONFIG
# ============================================================

DEVICE_NAME = "Tiresias_DK"

CHAR_UUID = "12345678-1234-5678-1234-56789abcdef1"

latest_quaternion = (1, 0, 0, 0)

# ============================================================
# QUATERNION -> EULER
# ============================================================

def quaternion_to_euler(w, x, y, z):

    sinr_cosp = 2 * (w * x + y * z)
    cosr_cosp = 1 - 2 * (x*x + y*y)
    roll = math.atan2(sinr_cosp, cosr_cosp)

    sinp = 2 * (w * y - z * x)

    if abs(sinp) >= 1:
        pitch = math.copysign(math.pi / 2, sinp)
    else:
        pitch = math.asin(sinp)

    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y*y + z*z)
    yaw = math.atan2(siny_cosp, cosy_cosp)

    return (
        math.degrees(roll),
        math.degrees(pitch),
        math.degrees(yaw),
    )

# ============================================================
# QUATERNION -> FORWARD VECTOR
# ============================================================

def quaternion_to_forward_vector(w, x, y, z):

    fx = 1 - 2*(y*y + z*z)
    fy = 2*(x*y + w*z)
    fz = 2*(x*z - w*y)

    return np.array([fx, fy, fz])

# ============================================================
# BLE CALLBACK
# ============================================================

def notification_handler(sender, data):

    global latest_quaternion

    if len(data) != 16:
        return

    latest_quaternion = struct.unpack("<ffff", data)

    roll, pitch, yaw = quaternion_to_euler(*latest_quaternion)

    print(
        f"\rYaw: {yaw:7.2f}° | "
        f"Pitch: {pitch:7.2f}° | "
        f"Roll: {roll:7.2f}°",
        end=""
    )

# ============================================================
# FIND DEVICE
# ============================================================

async def find_device():

    print("Scanning for BLE devices...\n")

    devices = await BleakScanner.discover(timeout=5.0)

    for d in devices:

        if d.name and DEVICE_NAME in d.name:

            print(f"Found device: {d.name}")
            print(f"Address: {d.address}")

            return d

    return None

# ============================================================
# BLE TASK
# ============================================================

async def ble_task():

    device = await find_device()

    if device is None:
        print("Device not found.")
        return

    print("\nConnecting...\n")

    async with BleakClient(device) as client:

        print("Connected.")
        print("Receiving head orientation...\n")

        await client.start_notify(
            CHAR_UUID,
            notification_handler
        )

        while True:
            await asyncio.sleep(0.01)

# ============================================================
# QT APPLICATION
# ============================================================

app = QApplication.instance()

if app is None:
    app = QApplication([])

# ============================================================
# 3D WINDOW
# ============================================================

view = gl.GLViewWidget()

view.show()

view.setWindowTitle("Head Tracking")

view.setCameraPosition(distance=4)

# Grid
grid = gl.GLGridItem()
grid.scale(1, 1, 1)

view.addItem(grid)

# ============================================================
# AXES
# ============================================================

axis = gl.GLAxisItem()

axis.setSize(1, 1, 1)

view.addItem(axis)

# ============================================================
# HEAD DIRECTION VECTOR
# ============================================================

line = gl.GLLinePlotItem(
    pos=np.array([[0,0,0], [1,0,0]]),
    width=5,
    antialias=True
)

view.addItem(line)

# ============================================================
# VISUAL UPDATE
# ============================================================

def update_visual():

    w, x, y, z = latest_quaternion

    direction = quaternion_to_forward_vector(w, x, y, z)

    pts = np.array([
        [0, 0, 0],
        direction
    ])

    line.setData(pos=pts)

# ============================================================
# QT TIMER
# ============================================================

timer = QTimer()

timer.timeout.connect(update_visual)

timer.start(16)   # ~60 FPS

# ============================================================
# RUN BLE
# ============================================================

asyncio.create_task(ble_task())

<Task pending name='Task-8' coro=<ble_task() running at /var/folders/dd/tqlfqw391q7glsdg0smpvh0m0000gn/T/ipykernel_32688/1652731733.py:113>>

Scanning for BLE devices...

Found device: Tiresias_DK
Address: C04C97BA-ED53-E435-DDE4-7055A36EF461

Connecting...

Connected.
Receiving head orientation...

Yaw:   -5.12° | Pitch:   -5.01° | Roll:    1.39°